In [1]:
import os
import pandas as pd
import joblib
import kagglehub
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier


# STEP 1: Dataset Choice & Preparation
print("Downloading dataset...")
path = kagglehub.dataset_download("alexteboul/diabetes-health-indicators-dataset")
csv_path = os.path.join(path, "diabetes_binary_health_indicators_BRFSS2015.csv")

print("Loading dataset into pandas...")
df = pd.read_csv(csv_path)

# The original dataset has 250k+ rows.
# We'll sample 20,000 rows to ensure fast training and avoid Streamlit memory limits.
df = df.sample(n=20000, random_state=42)

# Separate features (X) and target (y)
X = df.drop('Diabetes_binary', axis=1)
y = df['Diabetes_binary']

# Split into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Save the test set for Streamlit (Combining X_test and y_test back together)
print("Saving test_data.csv...")
test_data = pd.concat([X_test, y_test], axis=1)
test_data.to_csv('test_data.csv', index=False)

# Standardize the features (Important for KNN and Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# STEP 2: ML Models & Evaluation Metrics
# Create a folder to store the trained models
os.makedirs('model', exist_ok=True)
# Save the scaler as well, you'll need it for the Streamlit app!
joblib.dump(scaler, 'model/scaler.joblib')

# Initialize the 5 models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

# A dictionary to store results so we can print a nice comparison table at the end
results_table = []

print("\nTraining models and calculating metrics...\n")
print("-" * 60)

for name, model in models.items():
    print(f"Training {name}...")

    # Train the model
    model.fit(X_train_scaled, y_train)

    # Save the trained model to the 'model/' directory
    model_filename = f"model/{name.replace(' ', '_').lower()}.joblib"
    joblib.dump(model, model_filename)

    # Make predictions
    y_pred = model.predict(X_test_scaled)
    # Get probability of the positive class for AUC
    y_prob = model.predict_proba(X_test_scaled)[:, 1]

    # Calculate the 6 required metrics
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)

    # Store metrics for our summary table
    results_table.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "AUC": round(auc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1": round(f1, 4),
        "MCC": round(mcc, 4)
    })

print("-" * 60)
print("All models trained and saved successfully!\n")

# Display the final comparison table
results_df = pd.DataFrame(results_table)
print("EVALUATION METRICS COMPARISON TABLE:")
print(results_df.to_markdown(index=False))

Loading dataset into pandas...
Saving test_data.csv...

Training models and calculating metrics...

------------------------------------------------------------
Training Logistic Regression...
Training Decision Tree...
Training KNN...
Training Naive Bayes...
Training Random Forest...
------------------------------------------------------------
All models trained and saved successfully!

EVALUATION METRICS COMPARISON TABLE:
| Model               |   Accuracy |    AUC |   Precision |   Recall |     F1 |    MCC |
|:--------------------|-----------:|-------:|------------:|---------:|-------:|-------:|
| Logistic Regression |     0.8685 | 0.8409 |      0.5033 |   0.1442 | 0.2242 | 0.2176 |
| Decision Tree       |     0.802  | 0.6024 |      0.2838 |   0.3302 | 0.3053 | 0.1913 |
| KNN                 |     0.8605 | 0.7388 |      0.4362 |   0.2011 | 0.2753 | 0.2289 |
| Naive Bayes         |     0.7843 | 0.8054 |      0.3268 |   0.6015 | 0.4235 | 0.3263 |
| Random Forest       |     0.8682 | 0.